In [33]:
# =========================
# Standard library
# =========================
import os
import json
import time
import re
from typing import List, Dict, Union, Tuple, Optional
from gensim.corpora import Dictionary

# =========================
# Third-party
# =========================
import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.preprocessing import normalize
from sklearn.metrics import (
    accuracy_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
    f1_score,
)

from openai import OpenAI
from dotenv import load_dotenv

# =========================
# Local modules
# =========================
from get_embeddings import get_embeddings
from soft_kmeans import run_soft_kmeans
from hungarian_align import hungarian_align
from evaluation import evaluate_models

### [0.1.2.3] 클러스터 결과 load
0. 전처리 ~ 1. 임베딩 ~ 2. 클러스터 ~ 3. top words

In [34]:
# 1. 클러스터의 argmax값을 붙인 csv파일을 로드
df = pd.read_csv('/home/ys0660/2507Sub/textclustering/0915/Topicker/data/20NG/cluster_20ng.csv')
df.head()

,text,label_id,label_name,cleaned_text,cluster_label
0,i am sure some bashers of pens fans are pretty...,10,rec.sport.hockey,sure basher pen fan confused lack kind post re...,10
1,my brother is in the market for a high perform...,3,comp.sys.ibm.pc.hardware,brother market high performance video card loc...,3
2,the student of regional killings alias davidia...,17,talk.politics.mideast,student regional killing alias davidian davidi...,17
3,in article wayne smith writes think it s the s...,3,comp.sys.ibm.pc.hardware,scsi card dma transfer disk scsi card transfer...,3
4,1 i have an old jasmine drive which i cannot u...,4,comp.sys.mac.hardware,old jasmine drive new understanding driver mod...,3


In [35]:
# 0. 전처리 완료된 데이터 불러오기
def prepare_texts(df, col="cleaned_text"):
    # 각 문장을 토큰 리스트로 변환
    texts = []
    for s in df[col].astype(str):
        tokens = re.findall(r"[A-Za-z]+", s.lower())  # 알파벳만 추출, 소문자화
        texts.append(tokens)
    return texts

In [36]:
# 1. 임베딩 불러오기
embeddings = np.load('/home/ys0660/2507Sub/textclustering/0915/Topicker/data/20NG/text-embedding-3-small/embeddings.npz')

In [37]:
# 2. 클러스터 결과 불러오기
def load_soft_results(path: str):
    data = np.load(path, allow_pickle=True)
    P = data["P"]
    C = data["C"]
    y_pred_hard = data["y_pred_hard"]
    y_pred_aligned = data["y_pred_aligned"]
    mapping_arr = data["mapping"]
    mapping = {int(k): int(v) for k, v in mapping_arr}
    return P, C, mapping, y_pred_hard, y_pred_aligned

P_loaded, C_loaded, mapping_loaded, y_hard_loaded, y_aligned_loaded = load_soft_results("/home/ys0660/2507Sub/textclustering/0915/Topicker/data/20NG/text-embedding-3-small/soft_results.npz")

In [38]:
# 3. top words 가지고 오기
def load_topics_from_file(path: str, topn: int = 10):
    """
    path: topic_words.txt 파일 경로
    반환: topics = [["moral","argument","homosexual",...], ["image","jpeg",...], ...]
    """
    topics = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            # "Topic 0: moral, argument, homosexual, ..." 형태 가정
            line = line.strip()
            if not line:
                continue
            # "Topic", "번호:" 부분 제거 → 단어만 추출
            words = re.split(r"[:,\s]+", line)[2:]
            words = [w for w in words if w]
            topics.append(words[:topn])
    return topics

def prepare_texts(df, col="cleaned_text"):
    return [re.findall(r"[A-Za-z]+", str(s).lower()) for s in df[col].astype(str)]

def build_dict_corpus(texts):
    id2word = Dictionary(texts)
    corpus = [id2word.doc2bow(toks) for toks in texts]
    return id2word, corpus

topics = load_topics_from_file(
    "/home/ys0660/2507Sub/textclustering/0915/Topicker/data/20NG/text-embedding-3-small/topic_words.txt",
    topn=10
)
texts_tokenized = prepare_texts(df, "cleaned_text")
id2word, corpus = build_dict_corpus(texts_tokenized)

### [4] Quality Score

In [39]:
import numpy as np

def get_low_margin_indices(P, q=0.10):
    """
    P: (N, K) soft cluster probabilities
    q: 하위 비율 (0.10 = 10%)
    return: (indices, cutoff, n_selected)
    """
    sorted_probs = np.sort(P, axis=1)[:, ::-1]
    margins = sorted_probs[:, 0] - sorted_probs[:, 1]

    cutoff = np.quantile(margins, q)
    mask = margins <= cutoff
    indices = np.where(mask)[0]  # 인덱스 리스트 반환

    return indices, cutoff, mask.sum()

# ----------------------------
# 실행 (Hungarian 이후 예측 기준)
# ----------------------------
low10_indices, cutoff10, n10 = get_low_margin_indices(P_loaded, q=0.10)

print(f"Bottom 10% cutoff: {cutoff10:.4f}")
print(f"Selected {n10} samples as relabeling targets")
print("Example indices:", low10_indices[:20])  # 앞 20개만 보기


Bottom 10% cutoff: 0.0149
Selected 1824 samples as relabeling targets
Example indices: [ 37  41  42  52  54  63  76 104 106 107 110 125 127 130 135 142 165 173
 195 205]


### [5] LLM rationale

In [8]:
text_col = "cleaned_text" if "cleaned_text" in df.columns else "text"
assert text_col in df.columns, f"'{text_col}' column is required in df."

for col in ["llm_label", "llm_rationale", "llm_keywords", "llm_confidence", "llm_skip"]:
    if col not in df.columns:
        df[col] = np.nan

In [48]:
# 1차 재라벨한 결과
df = pd.read_csv("/home/ys0660/2507Sub/textclustering/0915/Topicker/data/20NG/cluster_20ng_llm.csv")

### [6] 하나의 임베딩 벡터로 만들어보기

In [10]:
from get_embeddings import get_embeddings

In [16]:
sub = df[df["llm_label"].notna()].copy()

In [19]:
# 리스트 변환
texts = sub["llm_label"].astype(str).tolist()

# 임베딩 계산
X_label = get_embeddings(texts)


Embedding: 100%|██████████| 1824/1824 [11:52<00:00,  2.56it/s] 


=== Embedding Summary ===
Total inputs   : 1824
OK             : 1824
Empty texts    : 0
API failures   : 0
Kept (non-zero): 1824


In [20]:
np.save("/home/ys0660/2507Sub/textclustering/0915/Topicker/src/X_label_1st.npy", X_label)

In [29]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import normalize
from sklearn.metrics import accuracy_score

from get_embeddings import get_embeddings
from soft_kmeans import run_soft_kmeans
from hungarian_align import hungarian_align

# -----------------------
# 설정
# -----------------------
K = 20                               # 20NG라면 20
USE_L2_NORMALIZED_MEAN = True        # True: L2 정규화 후 평균
USE_LABEL_PLUS_RATIONALE = False     # True면 llm_label + llm_rationale 결합
ALPHA = 0.5                          # (0~1) 원본/LLM 가중치. 0.5면 동일 가중

# df: 'label_id','text','llm_label','llm_rationale' 포함 가정
# embeddings: np.load(...) 결과 (키: "X")
# low10_indices: 재라벨 대상 인덱스 (np.array of int)
assert isinstance(low10_indices, (list, np.ndarray))
low10_indices = np.asarray(low10_indices, dtype=int)
assert np.all((0 <= low10_indices) & (low10_indices < len(df))), "low10_indices 범위 점검"

X_all_orig = embeddings["X"].astype(np.float32)           # (N, d)
y_true = df["label_id"].to_numpy()

# -----------------------
# 1) BEFORE
# -----------------------
P_bef, C_bef, y_pred_hard_bef = run_soft_kmeans(X_all_orig, n_clusters=K)
y_pred_aligned_bef, mapping_bef = hungarian_align(y_pred_hard_bef, y_true, P_bef, C_bef)

acc_before_targets = accuracy_score(
    y_true[low10_indices],
    y_pred_aligned_bef[low10_indices]
)
print(f"[Before] accuracy on relabel targets (n={len(low10_indices)}): {acc_before_targets:.4f}")

# -----------------------
# 2) LLM 임베딩 준비
# -----------------------
if USE_LABEL_PLUS_RATIONALE:
    texts = (
        df.iloc[low10_indices]["llm_label"].fillna("").astype(str) + " " +
        df.iloc[low10_indices]["llm_rationale"].fillna("").astype(str)
    ).tolist()
else:
    texts = df.iloc[low10_indices]["llm_label"].fillna("").astype(str).tolist()

X_llm = X_label #get_embeddings(texts)                              # (M, d)  M=len(low10_indices)
X_orig_targets = X_all_orig[low10_indices]                 # (M, d)
assert X_llm.shape == X_orig_targets.shape, "원본/LLM 임베딩 크기 불일치"

# -----------------------
# 3) 두 벡터 결합 (L2 정규화 + 가중 평균)
# -----------------------
if USE_L2_NORMALIZED_MEAN:
    X_a = normalize(X_orig_targets)                        # 행 단위 L2
    X_b = normalize(X_llm)
    # 가중 평균: (1-ALPHA)*원본 + ALPHA*LLM
    X_combined = normalize((1.0 - ALPHA) * X_a + ALPHA * X_b)
else:
    X_combined = (1.0 - ALPHA) * X_orig_targets + ALPHA * X_llm

# -----------------------
# 4) 전체 임베딩 업데이트 (대상만 교체)
# -----------------------
X_all_updated = X_all_orig.copy()
X_all_updated[low10_indices] = X_combined

# -----------------------
# 5) AFTER
# -----------------------
P_aft, C_aft, y_pred_hard_aft = run_soft_kmeans(X_all_updated, n_clusters=K)
y_pred_aligned_aft, mapping_aft = hungarian_align(y_pred_hard_aft, y_true, P_aft, C_aft)

acc_after_targets = accuracy_score(
    y_true[low10_indices],
    y_pred_aligned_aft[low10_indices]
)
print(f"[After ] accuracy on relabel targets (n={len(low10_indices)}): {acc_after_targets:.4f}")
print(f"[Delta ] improvement on targets: {acc_after_targets - acc_before_targets:+.4f}")

# (선택) df에 저장
df["1st_relabel"] = y_pred_aligned_aft.astype(int)
# 1차재라벨의 soft cluster 결과 저장
np.savez(
    "soft_results_1st_relabel.npz",
    P=P_aft,
    C=C_aft,
    y_pred_hard=y_pred_hard_aft,
    y_pred_aligned=y_pred_aligned_aft,
    mapping=np.array(list(mapping_aft.items()))
)



[Before] accuracy on relabel targets (n=1824): 0.1919
[After ] accuracy on relabel targets (n=1824): 0.3054
[Delta ] improvement on targets: +0.1135


In [ ]:
np.save("X_all_updated_1st.npy", X_all_updated)

In [26]:
# 전체 accuracy (전/후)
acc_before_all = accuracy_score(y_true, y_pred_aligned_bef)
acc_after_all  = accuracy_score(y_true, y_pred_aligned_aft)

print(f"[Overall] Accuracy before: {acc_before_all:.4f}")
print(f"[Overall] Accuracy after : {acc_after_all:.4f}")
print(f"[Overall] ΔAccuracy      : {acc_after_all - acc_before_all:+.4f}")

# 외부/내부 지표 한 번에 (evaluation.py 사용)
metrics_bef = evaluate_models(
    y_true,
    preds={"before": y_pred_aligned_bef},
    texts=df["text"].tolist(),
    topn=10
)
metrics_aft = evaluate_models(
    y_true,
    preds={"after": y_pred_aligned_aft},
    texts=df["text"].tolist(),
    topn=10
)

print("\n=== Overall metrics ===")
print("before:", metrics_bef["before"])
print("after :", metrics_aft["after"])


[Overall] Accuracy before: 0.5315
[Overall] Accuracy after : 0.6061
[Overall] ΔAccuracy      : +0.0746

=== Overall metrics ===
before: {'P1': 0.5549218535782835, 'ARI': 0.37680657914715565, 'NMI': 0.5543414591377103, 'TD': 0.54}
after : {'P1': 0.6103098437071566, 'ARI': 0.4374715567328458, 'NMI': 0.5823045700306145, 'TD': 0.505}


### 2번째

In [41]:
# 이전 loop에서의 soft kmeans 결과를 불러옴
P_loaded, C_loaded, mapping_loaded, y_hard_loaded, y_aligned_loaded = load_soft_results("soft_results_1st_relabel.npz")

In [42]:
low10_indices_2nd, cutoff10, n10 = get_low_margin_indices(P_loaded, q=0.10)

print(f"Bottom 10% cutoff: {cutoff10:.4f}")
print(f"Selected {n10} samples as relabeling targets")
print("Example indices:", low10_indices_2nd[:20])  # 앞 20개만 보기

Bottom 10% cutoff: 0.0005
Selected 1824 samples as relabeling targets
Example indices: [  5   8  12  27  49  53  58  61  63  73  76  80 101 117 141 142 148 162
 173 177]


In [46]:
# 이중에 1단계 재라벨과 겹치는 건?
arr1 = np.asarray(low10_indices)
arr2 = np.asarray(low10_indices_2nd)

# 교집합
common = np.intersect1d(arr1, arr2)

print(f"교집합 개수 : {len(common)} / 총 개수 {len(arr1)}")

교집합 개수 : 311 / 총 개수 1824


In [ ]:
# llm call 하는 py 만들어서 call한 다음, label을 df에다 붙이기
from get_labels import label_texts_only

# DataFrame → list
texts = df["text"][low10_indices_2nd].astype(str).tolist()

# label 값만 리스트로 받기
labels = label_texts_only(texts, model="gpt-4o")

In [57]:
# 새로운 label vector 추출
from get_embeddings import get_embeddings

# 리스트 변환
texts = labels

# 임베딩 계산
X_label_2nd = get_embeddings(texts)

np.save("/home/ys0660/2507Sub/textclustering/0915/Topicker/src/X_label_2nd.npy", X_label_2nd)

Embedding: 100%|██████████| 1824/1824 [11:15<00:00,  2.70it/s] 


=== Embedding Summary ===
Total inputs   : 1824
OK             : 1824
Empty texts    : 0
API failures   : 0
Kept (non-zero): 1824


In [58]:
from relabel_pipeline import (
    compute_baseline, prepare_llm_embeddings, combine_embeddings,
    run_cluster_and_align, evaluate_targets_acc, save_soft_results_npz
)

base = compute_baseline(X_all_updated, y_true, K=20, target_idx=low10_indices)
print(f"[Before] acc on targets: {base['acc_targets']:.4f}")

X_all_updated = combine_embeddings(
    X_all_orig, low10_indices_2nd, X_label_2nd,
    use_l2_normalized_mean=True, alpha=0.5
)

aft = run_cluster_and_align(X_all_updated, y_true, K=20)

accs = evaluate_targets_acc(
    y_true, base["y_aligned"], aft["y_aligned"], low10_indices_2nd
)
print(f"[After ] acc on targets: {accs['after']:.4f}")
print(f"[Delta ] improvement   : {accs['delta']:+.4f}")

[Before] acc on targets: 0.3054
[After ] acc on targets: 0.4150
[Delta ] improvement   : +0.1129


In [59]:
df["1st_relabel"] = base["y_aligned"].astype(int)
df["2nd_relabel"] = aft["y_aligned"].astype(int)

In [61]:
y_pred_aligned_bef = df['1st_relabel']
y_pred_aligned_aft = df['2nd_relabel']

In [62]:
# 전체 accuracy (전/후)
acc_before_all = accuracy_score(y_true, y_pred_aligned_bef)
acc_after_all  = accuracy_score(y_true, y_pred_aligned_aft)

print(f"[Overall] Accuracy before: {acc_before_all:.4f}")
print(f"[Overall] Accuracy after : {acc_after_all:.4f}")
print(f"[Overall] ΔAccuracy      : {acc_after_all - acc_before_all:+.4f}")

# 외부/내부 지표 한 번에 (evaluation.py 사용)
metrics_bef = evaluate_models(
    y_true,
    preds={"before": y_pred_aligned_bef},
    texts=df["text"].tolist(),
    topn=10
)
metrics_aft = evaluate_models(
    y_true,
    preds={"after": y_pred_aligned_aft},
    texts=df["text"].tolist(),
    topn=10
)

print("\n=== Overall metrics ===")
print("before:", metrics_bef["before"])
print("after :", metrics_aft["after"])


[Overall] Accuracy before: 0.6061
[Overall] Accuracy after : 0.6040
[Overall] ΔAccuracy      : -0.0021

=== Overall metrics ===
before: {'P1': 0.6103098437071566, 'ARI': 0.4374715567328458, 'NMI': 0.5823045700306145, 'TD': 0.505}
after : {'P1': 0.6087194954757335, 'ARI': 0.42793946422925866, 'NMI': 0.5843731310534136, 'TD': 0.5}


In [63]:
df.to_csv('/home/ys0660/2507Sub/textclustering/0915/Topicker/src/2nd_relabel.csv')

안좋아졌어......... 마진을 더 보수적으로 해야하나?

### 3번째

---

### loop을 돌지 말지 결정
중요한 문제이지만 일단은 신경쓰지 말고 2번, 3번, 4번, 5번 돌릴 것

In [28]:
import numpy as np
import pandas as pd

def cluster_entropy(labels):
    """
    labels: 리스트, 한 클러스터 내 라벨들
    """
    value_counts = pd.Series(labels).value_counts(normalize=True)
    probs = value_counts.values
    return -(probs * np.log(probs)).sum()

def compute_cluster_entropies(df, cluster_col, label_col):
    """
    df: DataFrame
    cluster_col: 클러스터 ID 컬럼명
    label_col: 라벨 컬럼명 (ex. 'llm_label')
    """
    results = {}
    for cluster_id, group in df.groupby(cluster_col):
        ent = cluster_entropy(group[label_col])
        results[cluster_id] = ent
    return results

# 개선 전후 엔트로피 계산
entropy_before = compute_cluster_entropies(df, "cluster_label", "label_id")
entropy_after  = compute_cluster_entropies(df, "1st_relabel", "label_id")

print("\n=== Cluster-wise Entropy Comparison ===")
for cid in sorted(entropy_before.keys()):
    bef = entropy_before[cid]
    aft = entropy_after.get(cid, np.nan)
    delta = aft - bef
    trend = "↓ 개선" if delta < 0 else ("↑ 악화" if delta > 0 else "→ 동일")
    print(f"Cluster {cid:2d}: before={bef:.4f}, after={aft:.4f}, Δ={delta:+.4f}  {trend}")

# 전체 평균 변화
avg_bef = np.mean(list(entropy_before.values()))
avg_aft = np.mean(list(entropy_after.values()))
print("\n=== Overall ===")
print(f"Mean entropy before={avg_bef:.4f}, after={avg_aft:.4f}, Δ={avg_aft-avg_bef:+.4f}")



=== Cluster-wise Entropy Comparison ===
Cluster  0: before=2.1098, after=2.0309, Δ=-0.0789  ↓ 개선
Cluster  1: before=1.3197, after=1.2378, Δ=-0.0819  ↓ 개선
Cluster  2: before=2.9267, after=2.8354, Δ=-0.0912  ↓ 개선
Cluster  3: before=1.4641, after=1.3657, Δ=-0.0984  ↓ 개선
Cluster  4: before=2.2321, after=1.7505, Δ=-0.4816  ↓ 개선
Cluster  5: before=1.1220, after=1.3546, Δ=+0.2327  ↑ 악화
Cluster  6: before=1.3891, after=1.1615, Δ=-0.2276  ↓ 개선
Cluster  7: before=0.7389, after=0.7152, Δ=-0.0236  ↓ 개선
Cluster  8: before=0.5702, after=0.5015, Δ=-0.0687  ↓ 개선
Cluster  9: before=0.2914, after=0.2722, Δ=-0.0192  ↓ 개선
Cluster 10: before=0.1639, after=0.1675, Δ=+0.0036  ↑ 악화
Cluster 11: before=0.3260, after=0.4384, Δ=+0.1123  ↑ 악화
Cluster 12: before=1.3911, after=1.4067, Δ=+0.0156  ↑ 악화
Cluster 13: before=0.6340, after=0.6336, Δ=-0.0004  ↓ 개선
Cluster 14: before=0.5974, after=0.5434, Δ=-0.0539  ↓ 개선
Cluster 15: before=1.0650, after=1.0620, Δ=-0.0030  ↓ 개선
Cluster 16: before=1.3941, after=1.3574, Δ=-0.0

In [ ]:
# 하위 10% 인덱스 뽑기
# 프롬프트 넣어서 자연어 label 받기
# 자연어 label과 산출물들을 2번째 자연어라벨이라는 의미의 칼럼에 새로 추가하기
# 자연어 label을 임베딩 벡터로 만들기
# 원본 텍스트와 2번째 자연어 라벨에 대한 벡터를 하나의 벡터로 만들기
# soft kmeans 수행한 후, 헝가리안 매치
# evaluation하기

In [ ]:
embed

In [ ]:
P_loaded, C_loaded, mapping_loaded, y_hard_loaded, y_aligned_loaded = load_soft_results("/home/ys0660/2507Sub/textclustering/0915/Topicker/data/20NG/text-embedding-3-small/soft_results.npz")

In [ ]:

# ===== 1) 프롬프트 넣어 v2 자연어 라벨/라쇼날 생성 & df 갱신 =====
for col in ["llm_label_v2", "llm_rationale_v2", "llm_confidence_v2", "llm_skip_v2"]:
    if col not in df.columns:
        df[col] = "" if ("confidence" not in col and "skip" not in col) else (0.0 if "confidence" in col else False)

for ridx in tqdm(target_idx, desc="LLM refresh (v2)"):
    txt = str(df.at[ridx, "text"]).strip()
    if not txt:
        df.at[ridx, "llm_label_v2"] = ""
        df.at[ridx, "llm_rationale_v2"] = ""
        df.at[ridx, "llm_confidence_v2"] = 0.0
        df.at[ridx, "llm_skip_v2"] = True
        continue
    j = call_llm_json(build_prompt(txt))
    df.at[ridx, "llm_label_v2"] = str(j.get("label", "")).strip()
    df.at[ridx, "llm_rationale_v2"] = str(j.get("rationale", "")).strip()
    df.at[ridx, "llm_confidence_v2"] = float(j.get("confidence", 0.0))
    df.at[ridx, "llm_skip_v2"] = bool(j.get("skip", False))

# ===== 2) 텍스트 준비: 원본(text) + v2 라벨 (없으면 기존 llm_label) =====
labels_new = (
    df.iloc[target_idx]["llm_label_v2"].fillna("").astype(str).str.strip().reset_index(drop=True)
)
labels_old = (
    df.iloc[target_idx]["llm_label"].fillna("").astype(str).str.strip().reset_index(drop=True)
)
orig_texts = (
    df.iloc[target_idx]["text"].fillna("").astype(str).reset_index(drop=True)
)

labels_new_np = labels_new.to_numpy()
labels_old_np = labels_old.to_numpy()
chosen_label = np.where(labels_new_np != "", labels_new_np, labels_old_np)

texts_v2 = [f"{t} [LABEL] {l}".strip() for t, l in zip(orig_texts.tolist(), chosen_label.tolist())]
assert len(texts_v2) == len(target_idx), "texts_v2 길이 불일치"

# ===== 3) v2 텍스트 임베딩 → 기존 임베딩과 결합 후 X_all_updated 생성 =====
X_llm_v2 = get_embeddings(texts_v2)           # (M, D)
X_orig_tg = X_all[target_idx]                  # (M, D)

assert X_llm_v2.shape == X_orig_tg.shape, "임베딩 차원/길이 불일치"
nz_mask = (np.linalg.norm(X_llm_v2, axis=1) > 0)

if USE_L2_NORMALIZED_MEAN:
    Xa = normalize(X_orig_tg)
    Xb = np.zeros_like(X_llm_v2)
    if nz_mask.any():
        Xb[nz_mask] = normalize(X_llm_v2[nz_mask])
    X_combined_slice = Xa.copy()
    X_combined_slice[nz_mask] = normalize(Xa[nz_mask] + Xb[nz_mask])
else:
    X_combined_slice = X_orig_tg.copy()
    X_combined_slice[nz_mask] = (X_orig_tg[nz_mask] + X_llm_v2[nz_mask]) / 2.0

X_all_updated = X_all.copy()
X_all_updated[target_idx] = X_combined_slice

print("[done] X_all_updated ready. shape:", X_all_updated.shape)

In [ ]:
# -----------------------
# BEFORE: 원본 임베딩
# -----------------------
K = 20
P_bef, C_bef, y_pred_hard_bef = run_soft_kmeans(embed, n_clusters=K)
y_pred_aligned_bef, mapping_bef = hungarian_align(y_pred_hard_bef, y_true, P_bef, C_bef)

metrics_bef = evaluate_models(
    y_true,
    preds={"before": y_pred_aligned_bef},
    texts=df["text"].tolist(),         # 내부지표(TD) 계산용
    topn=10,
    save_topic_path=None               # 원하면 경로 지정
)

# -----------------------
# AFTER: 업데이트 임베딩 (relabel 반영)
# -----------------------
P_aft, C_aft, y_pred_hard_aft = run_soft_kmeans(X_all_updated, n_clusters=K)
y_pred_aligned_aft, mapping_aft = hungarian_align(y_pred_hard_aft, y_true, P_aft, C_aft)

metrics_aft = evaluate_models(
    y_true,
    preds={"after": y_pred_aligned_aft},
    texts=df["text"].tolist(),
    topn=10,
    save_topic_path=None
)

# -----------------------
# 결과 출력 (전체 / 타겟 subset)
# -----------------------
print("=== Overall metrics ===")
print("before:", metrics_bef["before"])
print("after :", metrics_aft["after"])

# delta 요약
delta = {
    k: (metrics_aft["after"][k] - metrics_bef["before"][k]
        if (metrics_aft["after"][k] is not None and metrics_bef["before"][k] is not None)
        else None)
    for k in metrics_bef["before"].keys()
}
print("delta :", delta)

# (옵션) 재라벨 대상 subset 정확도만 별도 확인
acc_before_targets = accuracy_score(y_true[low10_indices], y_pred_aligned_bef[low10_indices])
acc_after_targets  = accuracy_score(y_true[low10_indices], y_pred_aligned_aft[low10_indices])
print(f"\nTarget (low10%) ACC: {acc_before_targets:.4f} -> {acc_after_targets:.4f} (Δ {acc_after_targets-acc_before_targets:+.4f})")